In [ ]:
# notebook 08_perturbation_manifold

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# load adata for use later
adata = sc.read_h5ad("../data/interim/07_adata_clustered.h5ad")

# in notebook 04 we ran sc.tl.rank_genes_groups(groupby="target")
# this runs DE for each (105 single-targets) CRISPRa vs. ctrl
# stored in adata.uns["rank_genes_groups"]

# then run sc.get.rank_genes_groups_df(adata, group="target_gene")
# as a for loop, for each gene, then get wilcoxon_de pd df
# cols= names (gene name), scores (wilcoxon), logfoldchanges (L2FC of that gene, perturbation vs ctrl), pvals, pvals_adj, target (CRISPRa)

# load from output of notebook 04
wilcoxon_de = pd.read_csv("../data/interim/04_wilcoxon_de_results.csv.gz")
print(wilcoxon_de.shape, wilcoxon_de.columns.tolist())
print(wilcoxon_de["target"].nunique(), "targets")

In [ ]:
# ── Build perturbation × gene L2FC signature matrix ──

# Pivot wilcoxon_de so it's one row per target, one column per gene, values = log2FC vs control
# here we are dropping info abou padj
# we will use this in clustering, so there just looking at effect size w/o padj
signature_matrix = wilcoxon_de.pivot(index="target", columns="names", values="logfoldchanges")

print(signature_matrix.shape)
signature_matrix.head()

In [ ]:
# the next step is PCA.  Can't handle NaN (if some comparisons were not conducted for some targets) but this is not the case
print("Total NaNs:", signature_matrix.isna().sum().sum())
print("Genes with any NaN:", (signature_matrix.isna().sum() > 0).sum(), "out of", signature_matrix.shape[1])

In [ ]:
# PCA with 105 rows and ~33k gene columns is extremely wide relative to rows.
# mathematically, this is fine - most genes will get 0 weight
# but reducing genes considered will help

# ── Restrict to genes with padj < 0.05 in at least N targets ──
sig_gene_counts = (
    wilcoxon_de[wilcoxon_de["pvals_adj"] < 0.05]
    .groupby("names")
    .size()
)

# N targets = 3
min_targets = 3  # gene must be a significant hit for at least 3 different targets
informative_genes = sig_gene_counts[sig_gene_counts >= min_targets].index

signature_matrix_filtered = signature_matrix[informative_genes]
print(signature_matrix_filtered.shape)
# results in 4602 retains genes/columns

# ── Z-score each gene across perturbations (same principle as scaling before PCA in notebook 07 —
# puts genes on comparable scales so high-variance genes don't dominate purely by magnitude) ──
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(signature_matrix_filtered.values)
scaled_df = pd.DataFrame(scaled, index=signature_matrix_filtered.index, columns=signature_matrix_filtered.columns)

# ── PCA + UMAP + clustering on perturbations (not cells) — same tools, tiny input this time ──
# scaled_df is just a 2D df, but make it into AnnData so you can use scanpy operations on it
pert_adata = ad.AnnData(scaled_df)

sc.tl.pca(pert_adata, svd_solver="arpack", n_comps=min(30, pert_adata.shape[0] - 1))
sc.pp.neighbors(pert_adata, n_neighbors=10, n_pcs=15)
sc.tl.umap(pert_adata)
sc.tl.leiden(pert_adata, resolution=1.0, flavor="igraph", n_iterations=2, key_added="leiden")

pert_adata.obs["leiden"].value_counts()


In [ ]:
# ── UMAP of perturbations, labeled by target name (feasible now — only ~105 points, not 102K) ──

umap_coords = pert_adata.obsm["X_umap"]
targets = pert_adata.obs_names.tolist()
clusters = pert_adata.obs["leiden"].values

palette = sc.pl.palettes.default_20
cluster_colors = dict(zip(pert_adata.obs["leiden"].cat.categories, palette))
point_colors = [cluster_colors[c] for c in clusters]

fig, ax = plt.subplots(figsize=(11, 10))

# Plot per-cluster so each gets its own legend entry (plotting once with c=point_colors has no legend handles)
for cluster_id in pert_adata.obs["leiden"].cat.categories:
    mask = (pert_adata.obs["leiden"] == cluster_id).values
    ax.scatter(
        umap_coords[mask, 0], umap_coords[mask, 1],
        c=cluster_colors[cluster_id], s=40, edgecolor="black", linewidth=0.5,
        label=f"Cluster {cluster_id}",
    )

from adjustText import adjust_text
texts = [
    ax.text(umap_coords[i, 0], umap_coords[i, 1], targets[i], fontsize=6)
    for i in range(len(targets))
]
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4, alpha=0.6))

ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Perturbation-level manifold\n(clustered by DE signature similarity, not raw expression)")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=False, title="Leiden cluster")

fig.tight_layout()
fig.savefig("../results/figures/perturbation_manifold_UMAP.png", dpi=200, bbox_inches="tight")
plt.show()

Figure 7.  Wilcoxon was performed on each CRISPRa target cell set vs ctrl, L2FC and padj reported.  Put together into one df, then reduced number of genes considered to those with padj <0.05 in at least 3 comparisons (4602 genes remain).  PCA, clustering, UMAP reveal 5 Leiden clusters.  Genes labeled on UMAP are CRISPRa perturbations; nearby points are perturbations that resulted in a similar DE signature across 4602 genes.

In [ ]:
## from here we are starting an indpendent analysis to the above
## to look at whether different targets drive cells to end up in different cluster buckets

# ── Composition table: fraction of each target's cells in each cell-state cluster ──

# 15% is roughly a 40x enrichment over background

# crosstab builds a cross-tabulation counting (contingency table), for every combo of the 2 categorical cols you give it
# normalize="index" converts the raw counts into fractions within each row
composition = pd.crosstab(adata.obs["target"], adata.obs["clusterName1"], normalize="index")

control_baseline = composition.loc["control"]

pd.set_option("display.max_rows", None)
print(control_baseline.sort_values(ascending=False))

In [ ]:
# Fisher's exact per (target, cluster) pair; computes true probability directly so it stays valid for small/rare categories (clusters)
# comparing that target's cells in vs. out of a given cluster against control's cells in vs. out of that same cluster
# null hypothesis is no enrichment for cells being found in that cluster, by virtue of target identity

# 105 targets x 24 clusters = 2500 tests (hence fdr_bh correction)

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

control_counts = adata.obs.loc[adata.obs["target"] == "control", "clusterName1"].value_counts()
control_total = control_counts.sum()

single_targets = [t for t in composition.index if t != "control" and t in adata.obs.loc[adata.obs["perturbation_type"] == "single", "target"].unique()]

results = []
for target in single_targets:
    target_counts = adata.obs.loc[adata.obs["target"] == target, "clusterName1"].value_counts()
    target_total = target_counts.sum()

    for cluster in adata.obs["clusterName1"].cat.categories:
        a = target_counts.get(cluster, 0)                # target cells in this cluster
        b = target_total - a                               # target cells NOT in this cluster
        c = control_counts.get(cluster, 0)                  # control cells in this cluster
        d = control_total - c                               # control cells NOT in this cluster

        odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative="two-sided")
        results.append({
            "target": target, "cluster": cluster,
            "target_frac": a / target_total, "control_frac": c / control_total,
            "odds_ratio": odds_ratio, "pval": pval,
        })

enrichment = pd.DataFrame(results)
enrichment["padj"] = multipletests(enrichment["pval"], method="fdr_bh")[1]
enrichment = enrichment.sort_values("padj")

enrichment.head(20)

# the output for fisher's exact test is this df: enrichment.
# this will be the source for plotting heatmap - odds_ratio is proportional to enrichment in a particular cluster

In [ ]:
# Sanity check: does KLF1 show significant erythroid enrichment, matching what we already know?
print(enrichment[enrichment["target"] == "KLF1"].sort_values("padj").head(10))

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# heatmap showing the distribution of cells to different clusters (clusterName1)
# some perturbations drive DE strong enough to influence clustering i.e. CEBPA B and E drive cells to Myeloid-like 2
# ── Single heatmap, control as boxed top row, all values on log2 scale ──

sig_hits = enrichment[enrichment["padj"] < 0.05].copy()
sig_hits["log2_odds"] = np.log2(sig_hits["odds_ratio"].clip(lower=1e-3))  # avoid -inf for odds_ratio=0

# Keep targets that have at least one strong hit, to avoid an overwhelming, mostly-empty heatmap
strong_targets = sig_hits.loc[sig_hits["log2_odds"].abs() > 2, "target"].unique()
heatmap_data = enrichment[enrichment["target"].isin(strong_targets)].pivot(
    index="target", columns="cluster", values="odds_ratio"
)
heatmap_data = np.log2(heatmap_data.clip(lower=1e-3))

# Express control's own fractions as log2(fraction / control's own row mean) — puts it on a
# similar log2-ish scale to the odds-ratio rows, so it doesn't just read as a flat outlier color.
# Simpler/more honest option: show control as log2(fraction), clipped, same colormap.
control_row_vals = np.log2(control_baseline.reindex(heatmap_data.columns).clip(lower=1e-3))
control_row_vals.name = "control"

full_heatmap_data = pd.concat([control_row_vals.to_frame().T, heatmap_data])

fig, ax = plt.subplots(figsize=(14, max(6, (len(strong_targets) + 1) * 0.25)))
sns.heatmap(full_heatmap_data, cmap="RdBu_r", center=full_heatmap_data.median().median(), ax=ax,
            cbar_kws={"label": "log2(odds ratio vs. control)  /  log2(fraction) for control row"})

# Bold box around row 0 (control)
from matplotlib.patches import Rectangle
rect = Rectangle((0, 0), full_heatmap_data.shape[1], 1, fill=False, edgecolor="black", linewidth=3)
ax.add_patch(rect)

ax.set_title("Perturbation → cell-state enrichment\n(targets with ≥1 strong, significant hit; control shown boxed)")
fig.tight_layout()
fig.savefig("../results/figures/perturbation_cluster_enrichment_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

# I feel this should be in an earlier notebook - with the initial clustering and analysis
# put it there in the readme, but leaving here for now in code

In [ ]:
# a fork in the road: compare tow different analyses

# earlier, we did clustering "naively" on the whole dataset.  From that, we can look by target, at two points on the UMAP space,
# and ask how close they are.  This is a "score" - more close on UMAP space means more shared pattern in gene expression

# now we have the heatmap, which was made using the Fisher's exact test on the 
# Showing some targets drive cells to different buckets (clusters)

# from here we want to see if those enrichements in the heatmap correlate with the original cell clustering UMAP designations

# and look at protein interaction networks for genes in different clusters from the L2FC clustering
# this could help define convergent signaling

In [ ]:
# ── Save notebook 08 outputs for downstream use (notebook 09, etc.) ──

enrichment.to_csv("../data/interim/08_perturbation_cluster_enrichment.csv.gz", index=False)
composition.to_csv("../data/interim/08_cluster_composition.csv.gz")
signature_matrix.to_csv("../data/interim/08_signature_matrix.csv.gz")

# pert_adata has PCA/UMAP/Leiden results — save as h5ad like everything else
pert_adata.write("../data/interim/08_pert_adata_manifold.h5ad", compression="gzip")